# 01 - Exploratory Data Analysis

This notebook explores the MOEX stock data for volatility forecasting.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 1. Load Data

In [ ]:
# Path to dataset_final
DATA_DIR = Path("../../moex_discovery/data/dataset_final")
STOCKS_DIR = DATA_DIR / "01_stocks" / "candles_10m"

# List available tickers
tickers = [f.stem for f in STOCKS_DIR.glob("*.parquet")]
print(f"Available tickers ({len(tickers)}): {sorted(tickers)}")

In [ ]:
# Load sample ticker data
SAMPLE_TICKER = "SBER"
df = pl.read_parquet(STOCKS_DIR / f"{SAMPLE_TICKER}.parquet")
print(f"Shape: {df.shape}")
df.head()

## 2. Data Overview

In [ ]:
# Basic statistics
df.describe()

In [ ]:
# Date range
df_pd = df.to_pandas()
df_pd['begin'] = pd.to_datetime(df_pd['begin'])

print(f"Date range: {df_pd['begin'].min()} to {df_pd['begin'].max()}")
print(f"Total bars: {len(df_pd):,}")
print(f"Trading days: {df_pd['begin'].dt.date.nunique():,}")

## 3. Intraday Patterns

In [ ]:
# Calculate log returns
df_pd['log_return'] = np.log(df_pd['close'] / df_pd['close'].shift(1))
df_pd['hour'] = df_pd['begin'].dt.hour

# Intraday volatility pattern
hourly_vol = df_pd.groupby('hour')['log_return'].apply(lambda x: np.sqrt((x**2).mean()))

fig, ax = plt.subplots(figsize=(10, 5))
hourly_vol.plot(kind='bar', ax=ax)
ax.set_xlabel('Hour')
ax.set_ylabel('Average Volatility')
ax.set_title(f'{SAMPLE_TICKER} - Intraday Volatility Pattern')
plt.tight_layout()
plt.show()

## 4. Daily Realized Volatility

In [ ]:
# Calculate daily RV
df_pd['date'] = df_pd['begin'].dt.date
daily_rv = df_pd.groupby('date')['log_return'].apply(lambda x: (x**2).sum())
daily_rv = daily_rv.reset_index()
daily_rv.columns = ['date', 'rv']
daily_rv['date'] = pd.to_datetime(daily_rv['date'])

print(f"Daily RV statistics:")
print(daily_rv['rv'].describe())

In [ ]:
# Plot RV time series
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(daily_rv['date'], daily_rv['rv'], linewidth=0.8)
ax.set_xlabel('Date')
ax.set_ylabel('Realized Volatility')
ax.set_title(f'{SAMPLE_TICKER} - Daily Realized Volatility')
plt.tight_layout()
plt.show()

## 5. Volatility Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Raw RV distribution
axes[0].hist(daily_rv['rv'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('RV')
axes[0].set_ylabel('Frequency')
axes[0].set_title('RV Distribution')

# Log RV distribution
axes[1].hist(np.log(daily_rv['rv'] + 1e-10), bins=50, edgecolor='black', alpha=0.7)
axes[1].set_xlabel('log(RV)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Log RV Distribution')

plt.tight_layout()
plt.show()

## 6. Autocorrelation

In [ ]:
from pandas.plotting import autocorrelation_plot

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# RV autocorrelation
pd.plotting.autocorrelation_plot(daily_rv['rv'].dropna(), ax=axes[0])
axes[0].set_title('RV Autocorrelation')
axes[0].set_xlim(0, 100)

# Log RV autocorrelation
pd.plotting.autocorrelation_plot(np.log(daily_rv['rv'] + 1e-10).dropna(), ax=axes[1])
axes[1].set_title('Log RV Autocorrelation')
axes[1].set_xlim(0, 100)

plt.tight_layout()
plt.show()

## 7. Cross-Ticker Analysis

In [ ]:
# Load RV for all tickers
all_rv = {}

for ticker in tickers[:5]:  # First 5 tickers
    try:
        df_t = pl.read_parquet(STOCKS_DIR / f"{ticker}.parquet").to_pandas()
        df_t['begin'] = pd.to_datetime(df_t['begin'])
        df_t['log_return'] = np.log(df_t['close'] / df_t['close'].shift(1))
        df_t['date'] = df_t['begin'].dt.date
        
        rv = df_t.groupby('date')['log_return'].apply(lambda x: (x**2).sum())
        all_rv[ticker] = rv
    except Exception as e:
        print(f"Error loading {ticker}: {e}")

rv_df = pd.DataFrame(all_rv)
print(f"Cross-ticker RV correlation:")
rv_df.corr()

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(rv_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=ax)
ax.set_title('RV Cross-Correlation')
plt.tight_layout()
plt.show()

## 8. Summary Statistics

In [ ]:
# Summary for all tickers
summary = []

for ticker in tickers:
    try:
        df_t = pl.read_parquet(STOCKS_DIR / f"{ticker}.parquet").to_pandas()
        df_t['begin'] = pd.to_datetime(df_t['begin'])
        
        summary.append({
            'ticker': ticker,
            'start_date': df_t['begin'].min().date(),
            'end_date': df_t['begin'].max().date(),
            'n_bars': len(df_t),
            'n_days': df_t['begin'].dt.date.nunique(),
        })
    except Exception as e:
        print(f"Error: {ticker} - {e}")

summary_df = pd.DataFrame(summary)
summary_df